# Diabetic Retinopathy using MobileNet 

## 1. Import library

In [44]:
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from sklearn.model_selection import train_test_split

## 2. Config

In [45]:
CONFIG = {
    "pkl_path": "../../../data/processed/aptos/sample_50/sample_dataset.pkl",
    "num_classes": 5,               # tự động lấy từ data['class_names']
    "input_size": 300,              # resize về 224 cho MobileNet
    "batch_size": 8,
    "num_epochs": 30,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "test_split": 0.2,
    "val_split": 0.1,               # tách từ phần train
    "checkpoint_path": "./mobilenet_best.pth",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

In [46]:
class RetinopathyDataset(Dataset):
    """
    Dataset wrap numpy array từ file .pkl.
    data['data']  : (N, H, W, 3)  uint8
    data['labels']: (N,)          int
    """
    def __init__(self, X: np.ndarray, y: np.ndarray, transform=None):
        self.X = X          # (N, H, W, 3)
        self.y = y          # (N,)
        self.transform = transform
 
    def __len__(self):
        return len(self.X)
 
    def __getitem__(self, idx):
        # chuyển (H,W,3) numpy → PIL → tensor
        from PIL import Image
        img = Image.fromarray(self.X[idx].astype(np.uint8))
        if self.transform:
            img = self.transform(img)
        label = int(self.y[idx])
        return img, label

In [47]:
def get_transforms(input_size: int):
    """Transform riêng cho train (có augment) và val/test."""
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
 
    train_tf = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
 
    val_tf = transforms.Compose([
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
 
    return train_tf, val_tf

In [48]:
def get_dataloaders(cfg: dict):
    """
    Load data từ file .pkl, split train/val/test, trả về 3 DataLoader.
    Cấu trúc pkl: dict với keys 'data', 'labels', 'class_names'
    """
    with open(cfg['pkl_path'], "rb") as f:
        data = pickle.load(f)
        X = data['data']
        y = data['labels']
        class_names = data['class_names']
        cfg['num_classes'] = len(class_names)
    print(f"[DATA] Tổng mẫu: {len(X)} | Số lớp: {cfg['num_classes']} " f"| Classes: {class_names}")
    print(f"[DATA] Shape ảnh gốc: {X.shape}")
    X_train, X_test, y_train, y_test = train_test_split(
        X, 
        y, 
        test_size=cfg['test_split'],
        random_state=42,
        stratify=y
    )
    # --- Tách val từ train ---
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train,
        test_size=cfg["val_split"],
        random_state=42,
        stratify=y_train,
    )
    print(f"[DATA] Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    train_tf, val_tf = get_transforms(cfg['input_size'])
    train_set = RetinopathyDataset(X_train, y_train, transform=train_tf)
    val_set = RetinopathyDataset(X_val, y_val, transform=val_tf)
    test_set = RetinopathyDataset(X_test, y_test, transform=val_tf)
    train_loader = DataLoader(train_set, batch_size=cfg["batch_size"], shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=cfg['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=cfg['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
    return train_loader, val_loader, test_loader, class_names

## 3. Model Definition

In [49]:
class DepthwiseSeparableConv(nn.Module):
    """
    Depthwise Separable Convolution:
      - Depthwise conv  : mỗi channel xử lý độc lập (groups=in_channels)
      - Pointwise conv  : 1x1 conv để kết hợp lại
      - Batch Norm + ReLU sau mỗi bước
    """
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3,
                      stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

In [50]:
class MobileNetV1(nn.Module):
    """
    MobileNet v1 — kiến trúc gốc từ bài báo Howard et al. 2017.
 
    Backbone:
      Conv / s2
      DW+PW x 13 (với các stride=2 tại các bước downsample)
      AvgPool → FC
    """
    def __init__(self, num_classes: int = 1000):
        super().__init__()
 
        # Lớp conv đầu tiên (standard)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
 
        # 13 Depthwise Separable Conv blocks
        # Format: (in_ch, out_ch, stride)
        dw_config = [
            (32,   64,  1),
            (64,  128,  2),
            (128, 128,  1),
            (128, 256,  2),
            (256, 256,  1),
            (256, 512,  2),
            *[(512, 512, 1)] * 5,   # 5 lần lặp stride=1
            (512, 1024, 2),
            (1024, 1024, 1),
        ]
 
        self.layers = nn.Sequential(
            *[DepthwiseSeparableConv(ic, oc, s) for ic, oc, s in dw_config]
        )
 
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1024, num_classes)
 
        self._init_weights()
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.layers(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [51]:
# training loop 
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0,0,0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += images.size(0)
    return total_loss/total, 100*correct/total 

In [52]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
 
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
 
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += images.size(0)
 
    return total_loss / total, 100.0 * correct / total

In [53]:
def train(cfg: dict):
    device = cfg["device"]
    print(f"[INFO] Sử dụng thiết bị: {device}")
 
    # Data
    train_loader, val_loader, test_loader, class_names = get_dataloaders(cfg)
 
    # Model
    model = MobileNetV1(num_classes=cfg["num_classes"]).to(device)
    print(f"[INFO] Tổng tham số: "
          f"{sum(p.numel() for p in model.parameters()):,}")
 
    # Loss & Optimizer
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )
 
    # LR Scheduler: CosineAnnealing
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["num_epochs"], eta_min=1e-6
    )
 
    best_val_acc = 0.0
 
    for epoch in range(1, cfg["num_epochs"] + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()
 
        print(
            f"Epoch [{epoch:>3}/{cfg['num_epochs']}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%"
        )
 
        # Lưu checkpoint tốt nhất
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc": val_acc,
            }, cfg["checkpoint_path"])
            print(f"  ✔ Đã lưu checkpoint (Val Acc: {val_acc:.2f}%)")
 
    print(f"\n[INFO] Huấn luyện xong. Best Val Acc = {best_val_acc:.2f}%")
    return model, test_loader, class_names

## 4. Evaluation & Inference

In [54]:
def load_checkpoint(path: str, cfg: dict) -> MobileNetV1:
    """Load model từ checkpoint đã lưu."""
    model = MobileNetV1(num_classes=cfg["num_classes"])
    ckpt = torch.load(path, map_location=cfg["device"])
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(cfg["device"])
    print(f"[INFO] Đã load checkpoint từ epoch {ckpt['epoch']} "
          f"(Val Acc lúc lưu: {ckpt['val_acc']:.2f}%)")
    return model
 
 
def final_test(model, test_loader, cfg: dict):
    """Đánh giá trên test set."""
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate(model, test_loader, criterion, cfg["device"])
    print(f"\n[RESULT] Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")
 
 
def predict_single(model, image_tensor: torch.Tensor,
                   class_names: list, device: str) -> str:
    """
    Dự đoán 1 ảnh.
    image_tensor: Tensor shape (1, C, H, W) đã normalize
    """
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.to(device)
        logits = model(image_tensor)
        prob = F.softmax(logits, dim=1)
        confidence, pred_idx = prob.max(1)
    label = class_names[pred_idx.item()]
    print(f"[Predict] {label} — confidence: {confidence.item():.2%}")
    return label

In [55]:
model,test_loader, class_names = train(CONFIG)

[INFO] Sử dụng thiết bị: cuda
[DATA] Tổng mẫu: 250 | Số lớp: 5 | Classes: ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferate_DR']
[DATA] Shape ảnh gốc: (250, 300, 300, 3)
[DATA] Train: 180 | Val: 20 | Test: 50
[INFO] Tổng tham số: 3,212,101


AttributeError: partially initialized module 'torch._inductor' has no attribute 'custom_graph_pass' (most likely due to a circular import)